# B3. Test the model runner on 10 rows

This notebook runs `src/run_model.py` for all three models on a fake manifest of 10 photos and checks that each JSONL file opens with `pandas.read_json(lines=True)`.

**Before you press Run all:**

1. Settings, Accelerator: **GPU T4 x1**. Settings, Internet: **on**.
2. Add-ons, Secrets: add `GITHUB_TOKEN` (a read-only GitHub token for this repo) and tick it for this notebook.

The token is only used to clone the repo. It is removed from the clone right after and never printed.

In [1]:
!pip install -q -U 'transformers>=4.57' accelerate bitsandbytes pillow num2words pyyaml
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, turn on the GPU in settings')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
torch 2.11.0+cu130 | transformers 5.22.2
GPU: Tesla T4


## 1. Get the code
Clones the private repo into `/tmp/repo`, which Kaggle does not save with the notebook.

In [2]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

cloned
c9f6d7e Add the model runner that writes one JSONL row per manifest row, with tests and a Kaggle notebook



## 2. A fake manifest of 10 photos
Ten tuning photos from `data/attack_set.csv`, downloaded from COCO, resized to a 512 px short side and saved as clean images. The manifest has the agreed columns.

In [3]:
import io, urllib.request
import pandas as pd
from PIL import Image

atk = pd.read_csv('data/attack_set.csv')
pick = atk[atk['split'] == 'tune'].head(10)
os.makedirs('/tmp/fake/images', exist_ok=True)
rows = []
for r in pick.itertuples():
    with urllib.request.urlopen('http://images.cocodataset.org/val2017/' + r.file_name, timeout=30) as f:
        im = Image.open(io.BytesIO(f.read())).convert('RGB')
    s = 512 / min(im.size)
    im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
    path = f'images/{r.image_id}_clean_none_none.png'
    im.save('/tmp/fake/' + path)
    rows.append({'image_id': r.image_id, 'cell': 'clean', 'setting': 'none', 'defence': 'none', 'path': path})
manifest = pd.DataFrame(rows)
manifest.to_csv('/tmp/fake/manifest.csv', index=False)
truth = dict(zip(pick['image_id'], pick['category']))
manifest

,image_id,cell,setting,defence,path
0,1503,clean,none,none,images/1503_clean_none_none.png
1,3661,clean,none,none,images/3661_clean_none_none.png
2,4795,clean,none,none,images/4795_clean_none_none.png
3,9914,clean,none,none,images/9914_clean_none_none.png
4,15254,clean,none,none,images/15254_clean_none_none.png
5,42178,clean,none,none,images/42178_clean_none_none.png
6,55528,clean,none,none,images/55528_clean_none_none.png
7,101762,clean,none,none,images/101762_clean_none_none.png
8,131386,clean,none,none,images/131386_clean_none_none.png
9,161978,clean,none,none,images/161978_clean_none_none.png


## 3. Run the three models
Each model runs in its own process, so its GPU memory is freed before the next one loads. The 7B model takes a few minutes to download.

In [4]:
MODELS = ['smolvlm2-2b', 'qwen3-vl-4b', 'llava-next-mistral-7b']
os.makedirs('/kaggle/working/runs', exist_ok=True)
for m in MODELS:
    print('=' * 70)
    !python src/run_model.py --model {m} --manifest /tmp/fake/manifest.csv --images-root /tmp/fake --out /kaggle/working/runs/b3_test_{m}.jsonl

smolvlm2-2b at 482adb537c02, config hash f58543b7b5398074a78efe6095f74f6f
processor_config.json: 100%|██████████████████| 67.0/67.0 [00:00<00:00, 398kB/s]
preprocessor_config.json: 100%|████████████████| 599/599 [00:00<00:00, 4.45MB/s]
config.json: 100%|█████████████████████████| 3.64k/3.64k [00:00<00:00, 16.5MB/s]
tokenizer_config.json: 100%|███████████████| 28.6k/28.6k [00:00<00:00, 92.5MB/s]
tokenizer.json: 100%|██████████████████████| 3.55M/3.55M [00:00<00:00, 68.3MB/s]
added_tokens.json: 100%|███████████████████| 4.74k/4.74k [00:00<00:00, 13.4MB/s]
special_tokens_map.json: 100%|█████████████████| 868/868 [00:00<00:00, 4.02MB/s]
model.safetensors.index.json: 100%|█████████| 63.9k/63.9k [00:00<00:00, 111MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 4.03GB            
Reconstructing (incomplete total..

## 4. Check the JSONL files
Each file must open with pandas, have 10 rows, have exactly the agreed fields and no repeated row.

In [5]:
FIELDS = ['image_id', 'cell', 'setting', 'defence', 'model', 'revision', 'prompt',
          'output', 'config_hash', 'seed', 'timestamp']
ok = True
for m in MODELS:
    df = pd.read_json(f'/kaggle/working/runs/b3_test_{m}.jsonl', lines=True)
    checks = {'10 rows': len(df) == 10, 'agreed fields': list(df.columns) == FIELDS,
              'no repeats': not df.duplicated(['image_id', 'cell', 'setting', 'defence']).any(),
              'one config hash': df['config_hash'].nunique() == 1, 'no empty answer': (df['output'].str.len() > 0).all()}
    ok &= all(checks.values())
    print(m, checks)
    for r in df.itertuples():
        print(f'    {truth[r.image_id]:14s} -> {r.output}')
print('OK' if ok else 'PROBLEM')

smolvlm2-2b {'10 rows': True, 'agreed fields': True, 'no repeats': True, 'one config hash': True, 'no empty answer': np.True_}
    laptop         -> computer.
    banana         -> Banana.
    tv             -> Cat.
    sandwich       -> sandwich.
    bowl           -> Food.
    train          -> Train.
    couch          -> remote.
    bicycle        -> Cat.
    airplane       -> Jet.
    skateboard     -> Skateboard.
qwen3-vl-4b {'10 rows': True, 'agreed fields': True, 'no repeats': True, 'one config hash': True, 'no empty answer': np.True_}
    laptop         -> computer
    banana         -> bananas
    tv             -> cat
    sandwich       -> burger
    bowl           -> lunchbox
    train          -> train
    couch          -> man
    bicycle        -> bicycle
    airplane       -> jet
    skateboard     -> skateboard
llava-next-mistral-7b {'10 rows': True, 'agreed fields': True, 'no repeats': True, 'one config hash': True, 'no empty answer': np.True_}
    laptop         -> C

## 5. Run one model again
Every row is already in its file, so the runner should print "every row is already done" and write nothing. This is a first look at the resume check in B4.

In [6]:
!python src/run_model.py --model qwen3-vl-4b --manifest /tmp/fake/manifest.csv --images-root /tmp/fake --out /kaggle/working/runs/b3_test_qwen3-vl-4b.jsonl
print('rows now:', len(pd.read_json('/kaggle/working/runs/b3_test_qwen3-vl-4b.jsonl', lines=True)))

qwen3-vl-4b at ebb281ec70b0, config hash 0930bfbd4205a016e28314617c890555
every row is already done, nothing to run
rows now: 10
